# Comparación completa -- PPO vs H0 vs H3 (día completo, 12 barcos, demanda 10%)

**Qué es esto, en una frase:** compara el agente PPO entrenado (`modelo_rl/ppo_dia_10pct_12barcos_reward_D_1M/`) contra las heurísticas H0 y H3 sobre las mismas 5 semillas de evaluación (1001-1005), mismo escenario exacto: 12 barcos, capacidad 30, demanda 10% de la población, día completo 06:00-24:00.

Genera: tablas comparativas agregadas, gráficas estáticas (PNG, para pegar directo en la presentación) e interactivas (HTML, Plotly), una animación GIF **y un video (.mp4)** de la mañana (06:00-08:00) con los tres paneles lado a lado -- mismo patrón que `heuristicas/notebooks/04_visualizacion_heuristicas.ipynb` para el GIF, y el mismo patrón que `comparacion/escalon_1_h0_h3_ppo/` para el video (un GIF no se puede pausar de forma confiable insertado en PowerPoint, un .mp4 sí). También agrega un reproductor interactivo paso a paso (en inglés, con panel de estado/decisiones/reservas) para cada política individual -- solo funciona con un kernel de Jupyter vivo.

**Metodología, en dos niveles (ver `README.md` de esta carpeta para el detalle):**
1. **Comparación oficial (agregada, 5 semillas).** H0/H3 se REUSAN tal cual de `heuristicas/outputs/resultados/resultados_h0_h1_h2_h3.csv` -- nunca recalculados. PPO se evalúa desde `modelo_rl/ppo_dia_10pct_12barcos_reward_D_1M/resultados_evaluacion_ppo.csv` (ya generado por `evaluar.py`, tampoco se recalcula aquí).
2. **Detalle ilustrativo (semilla 1001 únicamente).** Las gráficas de serie de tiempo y el GIF necesitan el `env` completo (historial paso a paso), que no está guardado en los CSV agregados. Para esto SÍ se vuelve a *correr* H0 y H3 (mismo código de `heuristicas/`, sin modificar una sola línea, sobre la misma demanda pre-generada `grupos_seed1001.csv`) y se vuelve a *evaluar* PPO para esa semilla (misma función `evaluar.correr_ppo`, sin reentrenar nada). Esto no cambia ningún número de la tabla oficial -- es solo para poder graficar/animar.

**No se modificó el simulador, las heurísticas, el modelo entrenado, ni GitHub** -- este notebook solo importa código existente y escribe archivos nuevos dentro de esta carpeta.

In [1]:
from __future__ import annotations

import shutil
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from matplotlib.animation import FuncAnimation, PillowWriter
from plotly.subplots import make_subplots
import imageio

BASE = Path.cwd().resolve().parent.parent.parent   # notebooks/ -> dia_completo/ -> comparacion/ -> raiz del repo
assert (BASE / "simulador").exists(), f"BASE mal calculado: {BASE}"

sys.path.insert(0, str(BASE / "simulador" / "src"))
sys.path.insert(0, str(BASE / "politica_base" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "comun" / "src"))
sys.path.insert(0, str(BASE / "heuristicas" / "h3_costo_global" / "src"))
sys.path.insert(0, str(BASE / "modelo_rl" / "ppo_dia_10pct_12barcos_reward_D_1M"))

from env import SimuladorBarcosBergen                 # noqa: E402
from estado import CODIGO_CORTO                        # noqa: E402
from politica_base import asignar_flota                # noqa: E402
from politica_h3 import asignar_flota_h3                # noqa: E402
import metricas as met                                  # noqa: E402
import visualizacion as viz                             # noqa: E402

import entrenar as E                                    # modelo_rl/ppo_dia_10pct_12barcos_reward_D_1M/entrenar.py -- solo se importa construir_entorno(), NO se reentrena nada
import evaluar as EV                                    # mismo folder -- solo se reusan construir_entorno_eval()/correr_ppo(), NO se recalcula la tabla oficial

from stable_baselines3 import PPO                       # noqa: E402
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize  # noqa: E402

OUT_DIR = Path.cwd().resolve().parent   # comparacion/dia_completo/
FIG_DIR = OUT_DIR / "figuras"
RES_DIR = OUT_DIR / "resultados"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

COLOR = {"H0": "#1f77b4", "H3": "#2ca02c", "PPO": "#e67e22"}
print("BASE =", BASE)
print("OUT_DIR =", OUT_DIR)

BASE = C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen
OUT_DIR = C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo


In [2]:
cfg_sim = yaml.safe_load(open(BASE / "simulador" / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BASE / "bergen-boats" / "config" / "instance.yaml", encoding="utf-8"))

nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BASE / "bergen-boats" / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)
capacidad = cfg_bb["flota"]["capacidad_pasajeros"]        # 30
nodo_inicial = cfg_bb["flota"]["nodo_inicial"]

NUM_BARCOS = 12
SEMILLAS_EVAL = cfg_sim["agente"]["evaluacion"]["semillas"]    # [1001..1005]
SEMILLA_ILUSTRATIVA = SEMILLAS_EVAL[0]                          # 1001 -- unica semilla usada en las graficas de detalle y el GIF

cfg_escalon = cfg_sim["escalones"]["escalon_dia_10pct"]
HORA_INI_MIN, HORA_FIN_MIN = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60   # [360, 1440] -- dia completo 06:00-24:00

DEMANDA_DIR = BASE / "politica_base" / "output" / "escalon_dia_10pct"
PPO_DIR = BASE / "modelo_rl" / "ppo_dia_10pct_12barcos_reward_D_1M"
H0_H3_CSV = BASE / "heuristicas" / "outputs" / "resultados" / "resultados_h0_h1_h2_h3.csv"

print(f"NUM_BARCOS={NUM_BARCOS}  capacidad={capacidad}  horas=[{HORA_INI_MIN},{HORA_FIN_MIN}] min  "
      f"semillas_eval={SEMILLAS_EVAL}  semilla_ilustrativa={SEMILLA_ILUSTRATIVA}")

NUM_BARCOS=12  capacidad=30  horas=[360,1440] min  semillas_eval=[1001, 1002, 1003, 1004, 1005]  semilla_ilustrativa=1001


## 1. Corridas de detalle (semilla 1001) -- H0, H3, PPO

Mismo código de `heuristicas/experimento_fleet_sweep.py::correr()` (H0/H3, sin modificar) y de `modelo_rl/ppo_dia_10pct_12barcos_reward_D_1M/evaluar.py::correr_ppo()` (PPO, sin modificar), corridos aquí una vez más -- solo para tener el `env` completo con `historial_estados` que las gráficas de detalle y el GIF necesitan.

In [3]:
def correr_heuristica(grupos_df: pd.DataFrame, nombre_politica: str):
    '''Copia exacta del bucle de heuristicas/experimento_fleet_sweep.py::correr()
    (codigo de heuristicas sin modificar, solo importado) -- se corre aqui
    UNICAMENTE para obtener el env completo (historial paso a paso) que
    hace falta para las graficas de detalle y el GIF. La tabla comparativa
    OFICIAL (seccion 2 mas abajo) no usa nada de esto: reusa
    resultados_h0_h1_h2_h3.csv tal cual, sin recalcular ningun numero.
    '''
    env = SimuladorBarcosBergen(
        grupos_df=grupos_df, matriz_tiempos=matriz_tiempos, nodos=nodos,
        num_barcos=NUM_BARCOS, capacidad_barco=capacidad,
        nodo_inicial=nodo_inicial, paso_tiempo_min=cfg_sim["paso_tiempo_min"],
        hora_inicio_min=HORA_INI_MIN, hora_fin_min=HORA_FIN_MIN,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )
    obs, info = env.reset(seed=cfg_sim["semilla"])
    reservas: dict = {}
    reservas_historial = [dict(reservas)]
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        if nombre_politica == "H0":
            decisiones = asignar_flota(libres, estado, matriz_tiempos, capacidad, cfg_sim["recompensa"])
        else:
            decisiones, reservas = asignar_flota_h3(libres, estado, matriz_tiempos, capacidad, reservas, cfg_sim["recompensa"])
        accion = np.array([env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0 for b in estado.barcos])
        obs, r, terminated, truncated, info = env.step(accion)
        reservas_historial.append(dict(reservas))
        if truncated or terminated:
            break
    return env, reservas_historial


grupos_1001 = pd.read_csv(DEMANDA_DIR / f"grupos_seed{SEMILLA_ILUSTRATIVA}.csv")
print(f"Demanda semilla {SEMILLA_ILUSTRATIVA} (H0/H3, pre-generada): {len(grupos_1001)} grupos, "
      f"{grupos_1001['tamano_grupo'].sum()} personas")

env_h0, _ = correr_heuristica(grupos_1001, "H0")
env_h3, reservas_hist_h3 = correr_heuristica(grupos_1001, "H3")
print("H0 y H3 corridos para semilla", SEMILLA_ILUSTRATIVA, "-- conservacion cuadra:",
      met.verificar_conservacion(env_h0)["cuadra"], met.verificar_conservacion(env_h3)["cuadra"])

Demanda semilla 1001 (H0/H3, pre-generada): 981 grupos, 8324 personas


H0 y H3 corridos para semilla 1001 -- conservacion cuadra: True True


In [4]:
print("Cargando modelo PPO y VecNormalize...")
model = PPO.load(str(PPO_DIR / "modelo_ppo.zip"))
venv_stats = VecNormalize.load(str(PPO_DIR / "vecnormalize.pkl"), DummyVecEnv([EV.construir_entorno_eval]))
venv_stats.training = False
venv_stats.norm_reward = False

env_ppo, reward_total_ppo = EV.correr_ppo(model, venv_stats, SEMILLA_ILUSTRATIVA)
print(f"PPO corrido para semilla {SEMILLA_ILUSTRATIVA} -- conservacion cuadra: "
      f"{met.verificar_conservacion(env_ppo)['cuadra']}")

# Chequeo de transparencia (no se esconde nada, ver README): H0/H3 leen la
# demanda de un CSV pre-generado (`grupos_seed1001.csv`), mientras que PPO
# la genera on-the-fly en `reset(seed=1001)` via `EntornoDemandaAleatoria`
# (mismo generador, `demand/src/llegadas.py`, pero otro camino de codigo) --
# si ambas rutas usan la misma semilla de verdad, el total de personas
# generadas DEBE coincidir exactamente. Se imprime sin asumir el resultado.
print(f"Personas generadas -- H0: {env_h0.total_unidades_generadas}, H3: {env_h3.total_unidades_generadas}, "
      f"PPO: {env_ppo.total_unidades_generadas} "
      f"({'MISMA realizacion de demanda' if env_h0.total_unidades_generadas == env_ppo.total_unidades_generadas else 'realizaciones DISTINTAS -- ver README, limitacion conocida'})")

Cargando modelo PPO y VecNormalize...


PPO corrido para semilla 1001 -- conservacion cuadra: True
Personas generadas -- H0: 8324, H3: 8324, PPO: 8324 (MISMA realizacion de demanda)


## 2. Comparación oficial agregada (5 semillas, H0/H3 sin recalcular)

Media ± desviación estándar sobre las 5 semillas de evaluación (1001-1005), 12 barcos, capacidad 30, demanda 10%. H0/H3 vienen de `heuristicas/outputs/resultados/resultados_h0_h1_h2_h3.csv`; PPO de `modelo_rl/ppo_dia_10pct_12barcos_reward_D_1M/resultados_evaluacion_ppo.csv`. **El reward NO se usa como criterio de comparación** (fórmulas distintas entre políticas -- ver sección 5).

In [5]:
heur = pd.read_csv(H0_H3_CSV)
heur_12 = heur[(heur["politica"].isin(["H0", "H3"])) & (heur["num_barcos"] == NUM_BARCOS)]
ppo_res = pd.read_csv(PPO_DIR / "resultados_evaluacion_ppo.csv")

METRICAS = [
    "pct_atendidas", "backlog_final_total", "espera_media_min", "espera_p95_min", "espera_max_min",
    "viaje_medio_min", "sistema_medio_min", "movimientos_totales", "movimientos_vacios",
    "movimientos_con_carga", "ocupacion_pct_capacidad", "pct_esperando_flota",
]
# sistema_p95_min / sistema_max_min: solo PPO las tiene calculadas (ver evaluar.py) --
# `experimento_fleet_sweep.py` no las guardo para H0/H3 originalmente.
METRICAS_EXTRA_PPO = ["sistema_p95_min", "sistema_max_min"]

filas = []
for pol, df_pol in [("H0", heur_12[heur_12.politica == "H0"]), ("H3", heur_12[heur_12.politica == "H3"]), ("PPO", ppo_res)]:
    fila = {"politica": pol, "n_semillas": len(df_pol)}
    for m in METRICAS:
        fila[f"{m}_mean"] = df_pol[m].mean()
        fila[f"{m}_std"] = df_pol[m].std()
    for m in METRICAS_EXTRA_PPO:
        fila[f"{m}_mean"] = df_pol[m].mean() if m in df_pol.columns else float("nan")
        fila[f"{m}_std"] = df_pol[m].std() if m in df_pol.columns else float("nan")
    filas.append(fila)

tabla_oficial = pd.DataFrame(filas).set_index("politica")
tabla_oficial.to_csv(RES_DIR / "tabla_comparativa_h0_h3_ppo_completa.csv")
tabla_oficial.round(3)

,n_semillas,pct_atendidas_mean,pct_atendidas_std,backlog_final_total_mean,backlog_final_total_std,espera_media_min_mean,espera_media_min_std,espera_p95_min_mean,espera_p95_min_std,espera_max_min_mean,...,movimientos_con_carga_mean,movimientos_con_carga_std,ocupacion_pct_capacidad_mean,ocupacion_pct_capacidad_std,pct_esperando_flota_mean,pct_esperando_flota_std,sistema_p95_min_mean,sistema_p95_min_std,sistema_max_min_mean,sistema_max_min_std
politica,,,,,,,,,,,,,,,,,,,,,
H0,5,99.583,0.168,34.2,14.132,6.178,0.171,15.706,0.552,28.026,...,564.6,9.529,16.310,0.482,26.285,1.275,NaN,NaN,NaN,NaN
H3,5,99.612,0.141,31.8,11.819,4.812,0.184,12.268,0.351,18.356,...,602.6,21.138,16.323,0.483,54.254,1.131,NaN,NaN,NaN,NaN
PPO,5,99.387,0.167,50.2,14.096,12.763,1.498,37.250,4.355,105.402,...,489.2,22.219,16.302,0.492,31.719,1.367,46.166,4.834,115.402,45.744


## 3. Gráficas comparativas -- barras con error (PNG listas para la presentación)

Media ± 1 desviación estándar entre las 5 semillas, para las tres políticas por igual (las tres tienen resultados por semilla disponibles).

In [6]:
def _grafico_barras(metricas: list[str], titulo: str, fname: str, figsize=(12, 4)):
    n = len(metricas)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    politicas = ["H0", "H3", "PPO"]
    for ax, m in zip(axes, metricas):
        medias = [tabla_oficial.loc[p, f"{m}_mean"] for p in politicas]
        stds = [tabla_oficial.loc[p, f"{m}_std"] for p in politicas]
        ax.bar(politicas, medias, yerr=stds, capsize=5, color=[COLOR[p] for p in politicas])
        ax.set_title(m, fontsize=10)
        ax.tick_params(axis="both", labelsize=9)
    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=200, bbox_inches="tight")
    plt.show()
    return fig


_grafico_barras(["pct_atendidas", "backlog_final_total"],
                 f"Servicio -- H0 vs H3 vs PPO (media +/- std, {len(SEMILLAS_EVAL)} semillas)",
                 "barras_servicio.png", figsize=(8, 4))

C:\Users\hfons\AppData\Local\Temp\ipykernel_32096\3267823796.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 800x400 with 2 Axes>

In [7]:
_grafico_barras(["espera_media_min", "espera_p95_min", "espera_max_min", "sistema_medio_min"],
                 f"Tiempos de espera y en sistema -- H0 vs H3 vs PPO (media +/- std, {len(SEMILLAS_EVAL)} semillas)",
                 "barras_tiempos.png", figsize=(15, 4))

C:\Users\hfons\AppData\Local\Temp\ipykernel_32096\3267823796.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1500x400 with 4 Axes>

In [8]:
_grafico_barras(["movimientos_totales", "movimientos_vacios", "movimientos_con_carga", "ocupacion_pct_capacidad", "pct_esperando_flota"],
                 f"Operación de flota -- H0 vs H3 vs PPO (media +/- std, {len(SEMILLAS_EVAL)} semillas)",
                 "barras_operacion_flota.png", figsize=(18, 4))

C:\Users\hfons\AppData\Local\Temp\ipykernel_32096\3267823796.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1800x400 with 5 Axes>

In [9]:
# Dashboard de una sola imagen -- las 12 metricas en una sola figura, para
# una sola diapositiva de resumen.
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
politicas = ["H0", "H3", "PPO"]
for ax, m in zip(axes.flat, METRICAS):
    medias = [tabla_oficial.loc[p, f"{m}_mean"] for p in politicas]
    stds = [tabla_oficial.loc[p, f"{m}_std"] for p in politicas]
    ax.bar(politicas, medias, yerr=stds, capsize=4, color=[COLOR[p] for p in politicas])
    ax.set_title(m, fontsize=10)
    ax.tick_params(labelsize=8)

handles = [mpatches.Patch(color=COLOR[p], label=p) for p in politicas]
fig.legend(handles=handles, loc="upper right", fontsize=11)
fig.suptitle(f"Dashboard comparativo -- H0 vs H3 vs PPO ({NUM_BARCOS} barcos, demanda 10%, "
             f"media +/- std sobre {len(SEMILLAS_EVAL)} semillas)", fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIG_DIR / "dashboard_comparativo_h0_h3_ppo.png", dpi=200, bbox_inches="tight")
plt.show()

C:\Users\hfons\AppData\Local\Temp\ipykernel_32096\3376029910.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Gráficas de detalle -- semilla 1001 (interactivas HTML + estáticas PNG)

Perfil de espera, ocupación de flota, % de cumplimiento por par O-D, desglose de recompensa y backlog final -- una por política, mismas funciones de `simulador/src/visualizacion.py` que usa el resto del proyecto (`viz.graficar_*`).

**Nota sobre el desglose de recompensa:** H0/H3 corrieron con la fórmula de recompensa de PRODUCCIÓN (`simulador/config/instance.yaml -> recompensa`), mientras que PPO corrió (entrenó y evaluó) con la fórmula **Reward D** (`agente.entrenamiento.recompensa_overrides`, sin tolerancia ni premio por entrega). **Las gráficas de desglose de recompensa de abajo NO son comparables entre políticas** -- cada una está en su propia escala/fórmula. Se incluyen solo porque son parte del reporte estándar de cada política individualmente, nunca para argumentar que una política "tiene mejor reward" que otra.

In [10]:
def _guardar_plotly(fig, nombre_base: str, width=1000, height=550):
    fig.write_html(FIG_DIR / f"{nombre_base}.html")
    fig.write_image(FIG_DIR / f"{nombre_base}.png", scale=2, width=width, height=height)


envs_detalle = {"H0": env_h0, "H3": env_h3, "PPO": env_ppo}

for nombre_pol, env in envs_detalle.items():
    fig = viz.graficar_perfil_espera(env)
    fig.update_layout(title=f"People waiting over time -- {nombre_pol} (semilla {SEMILLA_ILUSTRATIVA})")
    _guardar_plotly(fig, f"perfil_espera_{nombre_pol}")

    fig = viz.graficar_ocupacion_flota(env)
    fig.update_layout(title=f"Fleet occupancy over time -- {nombre_pol} (semilla {SEMILLA_ILUSTRATIVA})")
    _guardar_plotly(fig, f"ocupacion_flota_{nombre_pol}")

    fig = viz.graficar_heatmap_cumplimiento(env)
    fig.update_layout(title=f"% served by O-D pair -- {nombre_pol} (semilla {SEMILLA_ILUSTRATIVA})")
    _guardar_plotly(fig, f"heatmap_cumplimiento_{nombre_pol}")

    fig = viz.graficar_desglose_recompensa(env)
    formula = "Reward D" if nombre_pol == "PPO" else "produccion"
    fig.update_layout(title=f"Reward breakdown ({formula}) -- {nombre_pol} (semilla {SEMILLA_ILUSTRATIVA}) -- NO comparable entre políticas")
    _guardar_plotly(fig, f"desglose_recompensa_{nombre_pol}")

    fig = viz.graficar_sin_atender_al_final(env)
    fig.update_layout(title=f"Backlog at end of run -- {nombre_pol} (semilla {SEMILLA_ILUSTRATIVA})")
    _guardar_plotly(fig, f"backlog_{nombre_pol}")

print("Graficas de detalle (HTML + PNG) guardadas en", FIG_DIR)

Graficas de detalle (HTML + PNG) guardadas en C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras


## 5. Heatmap comparado lado a lado (% atendidas por par O-D)

In [11]:
fig_heatmaps = make_subplots(rows=1, cols=3, subplot_titles=["H0", "H3", "PPO"])
for i, (nombre_pol, env) in enumerate(envs_detalle.items(), start=1):
    df = met.metricas_por_par(env)
    m = pd.DataFrame(np.nan, index=nodos, columns=nodos)
    for _, r in df.iterrows():
        o, d = r["par"].split("->")
        m.loc[o, d] = r["pct_atendidas"]
    etiquetas = [CODIGO_CORTO[n] for n in nodos]
    texto = [[f"{v:.0f}%" if not np.isnan(v) else "" for v in fila] for fila in m.values]
    fig_heatmaps.add_trace(
        go.Heatmap(z=m.values, x=etiquetas, y=etiquetas, zmin=0, zmax=100, colorscale="RdYlGn",
                   text=texto, texttemplate="%{text}", showscale=(i == 3)),
        row=1, col=i,
    )
fig_heatmaps.update_layout(
    title=f"% atendidas por par O-D -- H0 vs H3 vs PPO (semilla {SEMILLA_ILUSTRATIVA})",
    template="plotly_white", height=420, width=1200,
)
_guardar_plotly(fig_heatmaps, "heatmap_cumplimiento_comparado", width=1200, height=420)
fig_heatmaps.show()

## 6. Animación GIF -- la mañana (06:00-08:00), H0 vs H3 vs PPO lado a lado

Mismo patrón que `heuristicas/notebooks/04_visualizacion_heuristicas.ipynb`: ventana de 2 horas (61 pasos de 2 min), 4 fps, mapa real de Bergen de fondo, barcos interpolados sobre la geometría real de las rutas navegables.

In [12]:
PASO_MIN = cfg_sim["paso_tiempo_min"]
N_FRAMES_MANANA = int(120 / PASO_MIN) + 1   # 61 -- ventana 06:00-08:00

gdf_nodos, gdf_rutas = viz.cargar_capas_mapa(cfg_bb, BASE / "bergen-boats" / "02_ruteo_navegable" / "output" / "rutas_navegables.geojson")
fondo = viz.descargar_fondo_mapa(gdf_nodos)

POLITICAS_GIF = ["H0", "H3", "PPO"]
historiales_manana = {
    "H0": env_h0.historial_estados[:N_FRAMES_MANANA],
    "H3": env_h3.historial_estados[:N_FRAMES_MANANA],
    "PPO": env_ppo.historial_estados[:N_FRAMES_MANANA],
}
reservas_h3_manana = reservas_hist_h3[:N_FRAMES_MANANA]


def _texto_reservas(reservas: dict) -> str:
    if not reservas:
        return "reservas activas: (ninguna)"
    return "reservas activas:\n" + "\n".join(f"  {bid} -> {o}->{d}" for bid, (o, d) in reservas.items())


fig, axes = plt.subplots(1, 3, figsize=(16, 6))
axes_por_politica = dict(zip(POLITICAS_GIF, axes))


def _actualizar_combinado(i):
    for pol in POLITICAS_GIF:
        ax = axes_por_politica[pol]
        frame = historiales_manana[pol][i]
        viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad, fondo=fondo)
        ax.set_title(pol, fontsize=13, fontweight="bold")
        if pol == "H3":
            ax.text(0.02, 0.02, _texto_reservas(reservas_h3_manana[i]), transform=ax.transAxes, fontsize=7,
                    va="bottom", ha="left", path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
    hh, mm = divmod(int(historiales_manana["H0"][i]["tiempo"]["minuto_del_dia"]), 60)
    fig.suptitle(f"H0 vs H3 vs PPO -- {hh:02d}:{mm:02d} (semilla {SEMILLA_ILUSTRATIVA}, {NUM_BARCOS} barcos, mañana 06:00-08:00)",
                 fontsize=13)


fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.02, wspace=0.05)
anim = FuncAnimation(fig, _actualizar_combinado, frames=N_FRAMES_MANANA, interval=1000 / 4)
out_gif = FIG_DIR / f"animacion_h0_h3_ppo_manana_semilla{SEMILLA_ILUSTRATIVA}.gif"
anim.save(out_gif, writer=PillowWriter(fps=4))
plt.close(fig)
print("GIF combinado guardado en", out_gif)

GIF combinado guardado en C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras\animacion_h0_h3_ppo_manana_semilla1001.gif


In [13]:
# GIF individual de PPO (mismo formato que los ya existentes de H0/H1/H2/H3
# en heuristicas/outputs/figuras/) -- no existia uno para PPO.
out_gif_ppo = FIG_DIR / f"animacion_PPO_semilla{SEMILLA_ILUSTRATIVA}_manana.gif"
viz.animar_corrida(historiales_manana["PPO"], gdf_nodos, gdf_rutas, matriz_tiempos, capacidad, out_gif_ppo, fps=5, fondo=fondo)
print("GIF individual PPO guardado en", out_gif_ppo)

# H0/H3 individuales: ya existen (mismo escenario -- 12 barcos, semilla 1001,
# ventana 06:00-08:00) en heuristicas/outputs/figuras/, generados por
# heuristicas/notebooks/04_visualizacion_heuristicas.ipynb -- se copian aqui
# (no se regeneran) para tener todo centralizado en una sola carpeta.
for pol in ["H0", "H3"]:
    src = BASE / "heuristicas" / "outputs" / "figuras" / f"animacion_{pol}.gif"
    if src.exists():
        shutil.copy2(src, FIG_DIR / f"animacion_{pol}_semilla{SEMILLA_ILUSTRATIVA}_manana.gif")
print("GIFs individuales H0/H3 (ya existentes, mismo escenario) copiados a", FIG_DIR)

GIF individual PPO guardado en C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras\animacion_PPO_semilla1001_manana.gif
GIFs individuales H0/H3 (ya existentes, mismo escenario) copiados a C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras


## 7. Videos (.mp4) -- la mañana, individuales y combinado

Reemplaza/complementa el GIF de arriba: un .mp4 SÍ se puede pausar insertado en PowerPoint (controles nativos de reproducción), un GIF no de forma confiable. Se genera con `imageio` + `imageio-ffmpeg` (mismo patrón que `comparacion/escalon_1_h0_h3_ppo/`), reusando `visualizacion.dibujar_frame` para cada cuadro -- nunca se modificó ese archivo salvo dos textos que se tradujeron a inglés ("waiting:" y el título del mapa), para que TODAS las animaciones del proyecto queden en inglés.

In [14]:
def animar_corrida_mp4(historial, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, out_path, fps=6, fondo=None,
                        figsize=(6.4, 6.4), dpi=100):
    '''Mismo dibujo que viz.animar_corrida (reusa viz.dibujar_frame sin
    modificarlo), pero escribe .mp4 (imageio+ffmpeg) en vez de .gif
    (PillowWriter) -- un video SI se puede pausar insertado en PowerPoint.
    figsize/dpi multiplos de 16 en pixeles (640x640) para que el encoder no
    tenga que rellenar/recortar.
    '''
    if fondo is None:
        fondo = viz.descargar_fondo_mapa(gdf_nodos)
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.94, bottom=0.01)
    writer = imageio.get_writer(str(out_path), fps=fps, codec="libx264", quality=8,
                                 macro_block_size=None, pixelformat="yuv420p")
    try:
        for frame in historial:
            viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, fondo=fondo)
            fig.canvas.draw()
            img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            writer.append_data(img)
    finally:
        writer.close()
        plt.close(fig)
    return out_path


def animar_combinado_mp4(historiales: dict, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, out_path,
                          titulo_extra: str, fps=6, fondo=None, figsize=(19.2, 6.4), dpi=100):
    '''Version combinada 1x3 (H0/H3/PPO lado a lado) como .mp4.'''
    if fondo is None:
        fondo = viz.descargar_fondo_mapa(gdf_nodos)
    politicas = list(historiales.keys())
    n_frames = len(historiales[politicas[0]])
    fig, axes = plt.subplots(1, len(politicas), figsize=figsize, dpi=dpi)
    fig.subplots_adjust(left=0.01, right=0.99, top=0.88, bottom=0.02, wspace=0.05)
    writer = imageio.get_writer(str(out_path), fps=fps, codec="libx264", quality=8,
                                 macro_block_size=None, pixelformat="yuv420p")
    try:
        for i in range(n_frames):
            for ax, pol in zip(axes, politicas):
                frame = historiales[pol][i]
                viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad_barco, fondo=fondo)
                ax.set_title(pol, fontsize=13, fontweight="bold")
                if pol == "H3":
                    ax.text(0.02, 0.02, _texto_reservas(reservas_h3_manana[i]), transform=ax.transAxes, fontsize=7,
                            va="bottom", ha="left", path_effects=[pe.withStroke(linewidth=2.5, foreground="white")])
            hh, mm = divmod(int(historiales[politicas[0]][i]["tiempo"]["minuto_del_dia"]), 60)
            fig.suptitle(f"{titulo_extra} -- {hh:02d}:{mm:02d}", fontsize=13)
            fig.canvas.draw()
            img = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
            writer.append_data(img)
    finally:
        writer.close()
        plt.close(fig)
    return out_path


for pol, historial in historiales_manana.items():
    out_mp4 = FIG_DIR / f"video_{pol}_semilla{SEMILLA_ILUSTRATIVA}_manana.mp4"
    animar_corrida_mp4(historial, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad, out_mp4, fps=6, fondo=fondo)
    print("Guardado:", out_mp4)

out_mp4_combinado = FIG_DIR / f"video_h0_h3_ppo_comparado_semilla{SEMILLA_ILUSTRATIVA}_manana.mp4"
animar_combinado_mp4(historiales_manana, gdf_nodos, gdf_rutas, matriz_tiempos, capacidad, out_mp4_combinado,
                      f"H0 vs H3 vs PPO (seed {SEMILLA_ILUSTRATIVA}, morning)", fps=6, fondo=fondo)
print("Guardado:", out_mp4_combinado)

Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras\video_H0_semilla1001_manana.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras\video_H3_semilla1001_manana.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras\video_PPO_semilla1001_manana.mp4


Guardado: C:\Users\hfons\Andes\OneDrive - Universidad de los Andes\NHH\NHH-Schedule-Free-Autonomous-Boats-in-Bergen\comparacion\dia_completo\figuras\video_h0_h3_ppo_comparado_semilla1001_manana.mp4


## 8. Interactive step-by-step player (English, live kernel only, full day)

Play/pause/slider + a side text panel (boat state, decisions, queues, step reward -- and for H3, the active reservations) that updates as you scroll minute by minute, across the FULL day (not just the morning window used for the GIF/video above). Same widget pattern as `visualizacion.reproductor_interactivo` (reused, not modified), rewritten in English and extended with reservations for H3. **Only works with a live Jupyter kernel** -- open this notebook in VS Code / Jupyter Lab and run these cells yourself; the buttons will not respond in this already-executed, saved copy. One cell per policy (not a shared loop) -- with several `ipywidgets` created in the same loop, only the last one ends up responding reliably.

In [15]:
import io
import ipywidgets as widgets
from IPython.display import Image, clear_output, display


def _state_text_en(env, frame: dict, reservas: dict | None = None) -> str:
    t = frame["tiempo"]["minuto_del_dia"]
    decisions = [e for e in env.log_eventos if e["tipo"] == "decision" and abs(e["minuto"] - t) < 1e-6]
    reward_step = next((r for r in env.log_recompensa if abs(r["minuto"] - t) < 1e-6), None)

    hh, mm = divmod(int(t), 60)
    lines = [f"=== {hh:02d}:{mm:02d} ===", "", "Boats:"]
    for i, b in enumerate(frame["barcos"]):
        dec = next((d["decision"] for d in decisions if d["barco_id"] == f"barco_{i}"), None)
        status = "idle" if b["libre"] else f"{b['origen']}->{b['destino']} ({b['min_para_llegar']:.0f} min left)"
        dec_txt = f", decided: {dec}" if dec is not None else ""
        lines.append(f"  boat_{i}: {status}, occupancy={b['ocupacion']}{dec_txt}")

    lines.append("")
    lines.append("Queues (pairs with people waiting):")
    any_queue = False
    for clave, v in frame["demanda"].items():
        if v["personas"] > 0:
            any_queue = True
            lines.append(f"  {clave}: {v['personas']} people, oldest waiting {v['espera_max']:.1f} min")
    if not any_queue:
        lines.append("  (none)")

    if reservas is not None:
        lines.append("")
        lines.append("H3 active reservations (boat -> queue it is committed to):")
        if not reservas:
            lines.append("  (none)")
        else:
            for bid, (o, d) in reservas.items():
                lines.append(f"  {bid} -> {o}->{d}")

    lines.append("")
    if reward_step:
        lines.append(f"Step reward: {reward_step['total']:.3f}")
        lines.append(f"  discomfort = {reward_step['incomodidad']:.3f}")
        lines.append(f"  movement   = {reward_step['movimiento']:.3f}")

    return "\n".join(lines)


def interactive_player(env, gdf_nodos, gdf_rutas, matriz_tiempos, title: str,
                        reservas_historial: list | None = None, fps: int = 4, fondo=None):
    '''English scrub player (play/pause/slider) with a side panel: boat
    state, decisions, queues and -- for H3, via reservas_historial -- the
    active reservations at that step. Same pattern as
    visualizacion.reproductor_interactivo (continuous_update=False so
    dragging the slider does not flood redraws), reused without modifying
    that file, rewritten in English. Only works with a LIVE Jupyter kernel.
    '''
    historial = env.historial_estados
    n_steps = len(historial)
    if fondo is None:
        fondo = viz.descargar_fondo_mapa(gdf_nodos)

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    plt.close(fig)

    out_map = widgets.Output()
    out_text = widgets.Output(layout=widgets.Layout(width="380px", border="1px solid #ccc", padding="8px"))

    slider = widgets.IntSlider(min=0, max=n_steps - 1, step=1, description="Step", continuous_update=False)
    play = widgets.Play(min=0, max=n_steps - 1, step=1, interval=int(1000 / fps), description="Play")
    widgets.jslink((play, "value"), (slider, "value"))

    back_btn = widgets.Button(description="< Prev step", layout=widgets.Layout(width="140px"))
    fwd_btn = widgets.Button(description="Next step >", layout=widgets.Layout(width="140px"))

    def _update(change):
        i = slider.value
        frame = historial[i]
        reservas = reservas_historial[i] if reservas_historial is not None else None
        viz.dibujar_frame(ax, frame, gdf_nodos, gdf_rutas, matriz_tiempos, env.capacidad_barco, fondo=fondo)
        ax.set_title(title, fontsize=12)
        with out_map:
            clear_output(wait=True)
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
            display(Image(data=buf.getvalue()))
        with out_text:
            clear_output(wait=True)
            print(_state_text_en(env, frame, reservas))

    def _step_fwd(b):
        slider.value = min(slider.value + 1, n_steps - 1)

    def _step_back(b):
        slider.value = max(slider.value - 1, 0)

    fwd_btn.on_click(_step_fwd)
    back_btn.on_click(_step_back)
    slider.observe(_update, names="value")
    _update(None)

    controls = widgets.HBox([back_btn, fwd_btn, play, slider])
    panel = widgets.HBox([out_map, out_text])
    ui = widgets.VBox([controls, panel])
    display(ui)
    return ui

In [16]:
print(f"=== Player: H0, seed {SEMILLA_ILUSTRATIVA} (full day) ===")
interactive_player(env_h0, gdf_nodos, gdf_rutas, matriz_tiempos, f"H0 -- seed {SEMILLA_ILUSTRATIVA}", fondo=fondo)

=== Player: H0, seed 1001 (full day) ===


In [17]:
print(f"=== Player: H3, seed {SEMILLA_ILUSTRATIVA} (full day) ===")
interactive_player(env_h3, gdf_nodos, gdf_rutas, matriz_tiempos, f"H3 -- seed {SEMILLA_ILUSTRATIVA}",
                    reservas_historial=reservas_hist_h3, fondo=fondo)

=== Player: H3, seed 1001 (full day) ===


In [18]:
print(f"=== Player: PPO, seed {SEMILLA_ILUSTRATIVA} (full day) ===")
interactive_player(env_ppo, gdf_nodos, gdf_rutas, matriz_tiempos, f"PPO -- seed {SEMILLA_ILUSTRATIVA}", fondo=fondo)

=== Player: PPO, seed 1001 (full day) ===


## 9. Veredicto honesto (automático, no basado en reward)

In [19]:
print("=" * 100)
print(f"COMPARACION FINAL -- H0 vs H3 vs PPO ({NUM_BARCOS} barcos, capacidad {capacidad}, demanda 10%, semillas {SEMILLAS_EVAL})")
print("=" * 100)
print(tabla_oficial[[f"{m}_mean" for m in METRICAS]].round(2).to_string())

print("\nMejor política por métrica (automático -- pct_atendidas: mayor mejor; el resto: menor mejor).")
print("El reward NO se usa aquí como criterio, por instrucción explícita del proyecto.\n")
for m in ["pct_atendidas"]:
    mejor = tabla_oficial[f"{m}_mean"].idxmax()
    print(f"  {m}: mejor = {mejor} ({tabla_oficial.loc[mejor, f'{m}_mean']:.2f})")
for m in ["backlog_final_total", "espera_media_min", "espera_p95_min", "espera_max_min", "sistema_medio_min"]:
    mejor = tabla_oficial[f"{m}_mean"].idxmin()
    print(f"  {m}: mejor (menor) = {mejor} ({tabla_oficial.loc[mejor, f'{m}_mean']:.2f})")

COMPARACION FINAL -- H0 vs H3 vs PPO (12 barcos, capacidad 30, demanda 10%, semillas [1001, 1002, 1003, 1004, 1005])
          pct_atendidas_mean  backlog_final_total_mean  espera_media_min_mean  espera_p95_min_mean  espera_max_min_mean  viaje_medio_min_mean  sistema_medio_min_mean  movimientos_totales_mean  movimientos_vacios_mean  movimientos_con_carga_mean  ocupacion_pct_capacidad_mean  pct_esperando_flota_mean
politica                                                                                                                                                                                                                                                                                                    
H0                     99.58                      34.2                   6.18                15.71                28.03                  9.79                   15.97                    1268.6                    704.0                       564.6                         16.31       

**Conclusión:** en este escenario (12 barcos, 10% de demanda, día completo), **PPO no supera a H0 ni a H3** en las métricas de experiencia del usuario (espera media/p95/máxima, tiempo en sistema, backlog). Es casi comparable en `pct_atendidas`. No se declara a PPO como "mejor" en ningún momento por tener un reward distinto o mayor -- el reward de PPO (Reward D) ni siquiera es comparable con la fórmula de producción que corrieron H0/H3 (ver nota de la sección 4). Ver `README.md` de esta carpeta para el detalle completo y las limitaciones conocidas.